# Philippine next-day heat-index dataset (Open-Meteo / ERA5)

Builds `ph_heat_index_next_day.csv` for the 149 Philippine cities, 2015-01-01 to 2025-12-29, and validates it.
The notebook only runs `build_ph_heat_index_dataset.py`; all logic lives there.

**Read this first - API quota.** The full job costs about **42,700 weighted Open-Meteo calls**. The free tier allows
10,000 per day, so a full build takes **about 5 daily runs**. Every finished city-year is cached, so each time you run the
run cell it resumes where it stopped and stops cleanly when the daily budget is used up. Keep the cache on Google Drive
(cell 1) so it survives runtime resets. With a commercial API key (cell 3) there is no daily limit.

Weather data by [Open-Meteo.com](https://open-meteo.com/) (CC BY 4.0); contains modified Copernicus Climate Change Service (ERA5) information. The free API is for non-commercial use.

In [ ]:
# 1) Where to keep the cache and the outputs. Google Drive is strongly recommended.
import os
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/ph_heat_index'
else:
    BASE = '/content/ph_heat_index'
WORK_DIR = f'{BASE}/cache'     # resumable API cache + run_log.txt
OUT_DIR  = f'{BASE}/output'    # the four final files
os.makedirs(WORK_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
print('cache :', WORK_DIR); print('output:', OUT_DIR)

In [ ]:
# 2) Get the code (skipped if the script is already in the working directory) and dependencies.
import os
REPO_URL = 'https://github.com/keyyum/heatcast-ph.git'
BRANCH   = 'claude/ph-heat-index-dataset-m8b8f5'      # change to 'main' once merged
if not os.path.exists('build_ph_heat_index_dataset.py'):
    if not os.path.exists('heatcast-ph'):
        !git clone --depth 1 -b $BRANCH $REPO_URL heatcast-ph
    %cd heatcast-ph
!pip -q install requests pandas numpy tqdm

In [ ]:
# 3) Optional: commercial Open-Meteo API key (leave empty for the free tier).
import os
os.environ['OPEN_METEO_API_KEY'] = ''   # paste a key here to use the customer endpoint with no client-side budget

In [ ]:
# 4) RUN. Safe to run repeatedly: finished chunks are never fetched again.
#    Exit code 2 / "STOPPED: daily budget" just means: run again after the budget resets (about 24h).
#    Use --max-cities 3 for a quick smoke test (writes a clearly-labelled SUBSET build).
!python build_ph_heat_index_dataset.py --work-dir "$WORK_DIR" --out-dir "$OUT_DIR"

In [ ]:
# 5) Look at the result (only exists once the build has completed).
import os, pandas as pd
path = f'{OUT_DIR}/ph_heat_index_next_day.csv'
if os.path.exists(path):
    df = pd.read_csv(path, parse_dates=['Date'])
    print(df.shape, df['Date'].min().date(), '->', df['Date'].max().date(), '| cities:', df['City'].nunique())
    display(df.head())
    print((df['HeatLevelTomorrow'].value_counts(normalize=True) * 100).round(2))
else:
    print('Dataset not built yet - run cell 4 again (the daily API budget may have been reached).')

In [ ]:
# 6) Optional: re-validate the saved file at any time (read-only).
!python build_ph_heat_index_dataset.py --validate-only "$OUT_DIR/ph_heat_index_next_day.csv"